# 🎯 Valorant Performance Intelligence: Exploratory Data Analysis & Modeling

This interactive notebook demonstrates the complete end-to-end data science workflow:
1. **Data Ingestion & Inspection:** Loading clean Valorant telemetry from SQLite and CSV.
2. **Feature Engineering:** Calculating tactical ratios (K/D, ACS/min, First Blood rate) and guarding against data leakage.
3. **Exploratory Data Analysis:** Analyzing agent meta, map balance, role ACS distributions, and rank progression.
4. **Machine Learning Model Training:** Benchmarking Logistic Regression, Random Forest, and XGBoost with Stratified 5-Fold Cross-Validation.
5. **Interpretability with SHAP:** Computing local and global feature attribution.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
project_root = Path("..").resolve()
data_path = project_root / "data" / "processed" / "valorant_features.parquet"

# Load processed telemetry
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} match telemetry rows with {len(df.columns)} columns.")
df.head()

## 1. Tactical Telemetry Overview & Summary Statistics

In [ ]:
key_metrics = ["kills", "deaths", "assists", "average_combat_score", "headshot_percentage", "first_bloods", "economy_rating", "kd_ratio"]
df[key_metrics].describe().T

## 2. Agent Win Rates and Tactical Role Distributions

In [ ]:
role_summary = df.groupby("role").agg(
    win_rate=("match_result", lambda s: (s.str.lower() == "win").mean() * 100),
    avg_kd=("kd_ratio", "mean"),
    avg_acs=("average_combat_score", "mean"),
    avg_first_bloods=("first_bloods", "mean"),
    avg_econ=("economy_rating", "mean"),
).round(2)
role_summary

## 3. Load Trained Model & Perform Live Inference with SHAP

In [ ]:
import joblib
import shap

model = joblib.load(project_root / "models" / "best_model.joblib")
preprocessor = joblib.load(project_root / "models" / "preprocessor.joblib")
explainer = joblib.load(project_root / "models" / "shap_explainer.joblib")

# Create sample input test point
sample = df.iloc[[0]][[
    "agent", "role", "map", "rank", "attack_defense_side",
    "kills", "deaths", "assists", "average_combat_score", "headshot_percentage",
    "first_bloods", "clutches", "economy_rating", "match_duration",
    "kd_ratio", "kda_ratio", "combat_efficiency", "first_blood_rate", "econ_efficiency"
]]

X_sample = preprocessor.transform(sample)
win_prob = model.predict_proba(X_sample)[0, 1]
print(f"Predicted Win Probability: {win_prob * 100:.2f}%")
print(f"Ground truth match result: {df.iloc[0]['match_result']}")